# 03 - Pedestrian Flow Analysis

This notebook converts anonymous bbox CSV rows into ground coordinates, links detections into anonymous tracks, estimates speed, and exports report plots and metric tables. It does not use images or video from normal data collection.

In [ ]:
from pathlib import Path

import pandas as pd

from pedflow.geometry import detections_to_ground, load_calibration
from pedflow.metrics import add_dwell_flags, estimate_speeds, grid_statistics, summarize_flow, track_summaries
from pedflow.plotting import write_standard_plots
from pedflow.tracking import filter_short_tracks, link_detections

DETECTIONS_CSV = Path("../data/detections/session.csv")
CALIBRATION_JSON = Path("../outputs/calibration.json")
OUTPUT_DIR = Path("../outputs/analysis")

CONFIDENCE_THRESHOLD = 0.6
MAX_MATCHING_SPEED_M_S = 4.5
CLOSE_AFTER_S = 1.0
MIN_TRACK_DURATION_S = 1.5
MIN_DETECTIONS = 4
SMOOTHING_ALPHA = 0.3
SPEED_WINDOW_S = 0.75
STOP_SPEED_THRESHOLD_M_S = 0.2
STOP_DURATION_THRESHOLD_S = 2.0
GRID_SIZE_M = 0.5

In [ ]:
detections = pd.read_csv(DETECTIONS_CSV)
calibration = load_calibration(CALIBRATION_JSON)

ground = detections_to_ground(
    detections,
    calibration,
    confidence_threshold=CONFIDENCE_THRESHOLD,
)
ground.head()

In [ ]:
tracks = link_detections(
    ground,
    max_matching_speed_m_s=MAX_MATCHING_SPEED_M_S,
    close_after_s=CLOSE_AFTER_S,
    smoothing_alpha=SMOOTHING_ALPHA,
)
tracks = filter_short_tracks(
    tracks,
    min_duration_s=MIN_TRACK_DURATION_S,
    min_detections=MIN_DETECTIONS,
)
tracks = estimate_speeds(tracks, window_s=SPEED_WINDOW_S)
tracks = add_dwell_flags(
    tracks,
    stop_speed_threshold_m_s=STOP_SPEED_THRESHOLD_M_S,
    stop_duration_threshold_s=STOP_DURATION_THRESHOLD_S,
)
tracks.head()

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ground.to_csv(OUTPUT_DIR / "detections_ground.csv", index=False)
tracks.to_csv(OUTPUT_DIR / "tracks.csv", index=False)
summaries = track_summaries(tracks)
summary = summarize_flow(tracks)
grid = grid_statistics(tracks, grid_size_m=GRID_SIZE_M)

summaries.to_csv(OUTPUT_DIR / "track_summaries.csv", index=False)
summary.to_csv(OUTPUT_DIR / "summary_metrics.csv", index=False)
grid.to_csv(OUTPUT_DIR / "grid_metrics.csv", index=False)

write_standard_plots(tracks, OUTPUT_DIR, grid_size_m=GRID_SIZE_M)
summary